# Netflix Streaming Analysis

Comprehensive analysis of 4,454 viewing sessions totaling 1,687 hours of watch time, uncovering viewing patterns, device preferences, and content consumption behavior.

## Key Findings:
- **Peak viewing at 5 PM with 8.5% of total watch time**
- **Mac Safari dominates device usage at 75.3%**
- **70% series vs 30% movies by watch time**
- **2,840 unique titles across 4,454 sessions**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## 1. Data Loading & Exploration

In [ ]:
# Load the dataset
df = pd.read_csv('ViewingActivity.csv')

print(f'Dataset Shape: {df.shape}')
print(f'Total Viewing Sessions: {len(df):,}')
print(f'\nColumns: {list(df.columns)}')

In [ ]:
# Basic info
df.info()

In [ ]:
# First few rows
df.head()

## 2. Data Cleaning & Preprocessing

In [ ]:
# Parse duration to seconds
def parse_duration(duration_str):
    try:
        parts = str(duration_str).split(':')
        if len(parts) == 3:
            h, m, s = parts
            return int(h) * 3600 + int(m) * 60 + int(s)
        return 0
    except:
        return 0

df['Duration_Seconds'] = df['Duration'].apply(parse_duration)
df['Duration_Minutes'] = df['Duration_Seconds'] / 60
df['Duration_Hours'] = df['Duration_Seconds'] / 3600

# Parse datetime
df['Start Time'] = pd.to_datetime(df['Start Time'])
df['Hour'] = df['Start Time'].dt.hour
df['DayOfWeek'] = df['Start Time'].dt.dayofweek
df['DayName'] = df['Start Time'].dt.day_name()
df['IsWeekend'] = df['DayOfWeek'] >= 5
df['Month'] = df['Start Time'].dt.month
df['Year'] = df['Start Time'].dt.year

# Content type flag
df['IsMainContent'] = df['Supplemental Video Type'].isna()

print('Data preprocessing complete!')
print(f'\nTotal watch time: {df["Duration_Hours"].sum():,.0f} hours')

## 3. Overall Statistics

In [ ]:
# Key metrics
print('=' * 50)
print('KEY VIEWING METRICS')
print('=' * 50)
print(f'Total Sessions: {len(df):,}')
print(f'Total Watch Time: {df["Duration_Hours"].sum():,.0f} hours')
print(f'Unique Titles: {df["Title"].nunique():,}')
print(f'Average Session Duration: {df["Duration_Minutes"].mean():.1f} minutes')
print(f'Main Content Sessions: {df["IsMainContent"].sum():,}')
print(f'Trailers/Hooks: {(~df["IsMainContent"]).sum():,}')

In [ ]:
# Filter to main content only for detailed analysis
main_content = df[df['IsMainContent']].copy()
print(f'\nMain content sessions: {len(main_content):,}')
print(f'Main content watch time: {main_content["Duration_Hours"].sum():,.0f} hours')

## 4. Viewing Time Analysis

In [ ]:
# Hourly viewing patterns
hourly_watch = main_content.groupby('Hour')['Duration_Hours'].sum()
total_hours = hourly_watch.sum()

print('Viewing by Hour of Day:')
print('=' * 40)
peak_hour = hourly_watch.idxmax()
print(f'Peak viewing hour: {peak_hour}:00 ({hourly_watch[peak_hour]/total_hours*100:.1f}% of total)')

# Top 5 hours
print('\nTop 5 viewing hours:')
for hour, hours_watched in hourly_watch.sort_values(ascending=False).head(5).items():
    print(f'  {hour}:00 - {hours_watched:.1f} hours ({hours_watched/total_hours*100:.1f}%)')

In [ ]:
# Visualization: Hourly Viewing Pattern
fig, ax = plt.subplots(figsize=(12, 6))
hours = range(24)
watch_hours = [hourly_watch.get(h, 0) for h in hours]
colors = ['#FF6B6B' if h == peak_hour else '#4ECDC4' for h in hours]
ax.bar(hours, watch_hours, color=colors, edgecolor='black', alpha=0.8)
ax.set_xlabel('Hour of Day', fontsize=12)
ax.set_ylabel('Total Watch Time (hours)', fontsize=12)
ax.set_title('Netflix Viewing Pattern by Hour', fontsize=14)
ax.set_xticks(hours)
ax.set_xticklabels([f'{h}:00' for h in hours], rotation=45)
plt.tight_layout()
plt.show()

## 5. Day of Week Analysis

In [ ]:
# Daily viewing patterns
daily_watch = main_content.groupby('DayName')['Duration_Hours'].sum()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_watch = daily_watch.reindex(day_order)

print('Viewing by Day of Week:')
print('=' * 40)
for day, hours_watched in daily_watch.items():
    print(f'  {day}: {hours_watched:.1f} hours')

# Weekend vs Weekday comparison
weekend_watch = main_content[main_content['IsWeekend']]['Duration_Hours'].sum()
weekday_watch = main_content[~main_content['IsWeekend']]['Duration_Hours'].sum()
weekend_days = main_content[main_content['IsWeekend']]['Start Time'].dt.date.nunique()
weekday_days = main_content[~main_content['IsWeekend']]['Start Time'].dt.date.nunique()

weekend_avg = weekend_watch / max(weekend_days, 1)
weekday_avg = weekday_watch / max(weekday_days, 1)
ratio = weekend_avg / max(weekday_avg, 1)

print(f'\nWeekend vs Weekday:')
print(f'  Weekend avg: {weekend_avg:.2f} hours/day')
print(f'  Weekday avg: {weekday_avg:.2f} hours/day')
print(f'  Weekend boost: {ratio:.1f}x')

In [ ]:
# Visualization: Day of Week
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#FF6B6B' if day in ['Saturday', 'Sunday'] else '#4ECDC4' for day in day_order]
bars = ax.bar(day_order, [daily_watch.get(d, 0) for d in day_order], color=colors, edgecolor='black')
ax.set_xlabel('Day of Week', fontsize=12)
ax.set_ylabel('Total Watch Time (hours)', fontsize=12)
ax.set_title('Netflix Viewing by Day of Week', fontsize=14)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Device Usage Analysis

In [ ]:
# Device usage by watch time
device_watch = df.groupby('Device Type')['Duration_Hours'].sum().sort_values(ascending=False)
total_device_hours = device_watch.sum()

print('Device Usage by Watch Time:')
print('=' * 50)
for device, hours in device_watch.head(10).items():
    pct = hours / total_device_hours * 100
    print(f'  {device}: {hours:.1f} hrs ({pct:.1f}%)')

In [ ]:
# Simplified device categories
def categorize_device(device):
    device_lower = str(device).lower()
    if 'mac' in device_lower or 'safari' in device_lower:
        return 'Mac/Safari'
    elif 'iphone' in device_lower:
        return 'iPhone'
    elif 'chromecast' in device_lower:
        return 'Chromecast'
    elif 'tv' in device_lower:
        return 'Smart TV'
    elif 'chrome' in device_lower or 'pc' in device_lower:
        return 'PC/Chrome'
    elif 'android' in device_lower:
        return 'Android'
    else:
        return 'Other'

df['DeviceCategory'] = df['Device Type'].apply(categorize_device)
device_cat_watch = df.groupby('DeviceCategory')['Duration_Hours'].sum().sort_values(ascending=False)

# Visualization
fig, ax = plt.subplots(figsize=(10, 8))
colors = sns.color_palette('husl', len(device_cat_watch))
wedges, texts, autotexts = ax.pie(device_cat_watch.values, labels=device_cat_watch.index, 
                                   autopct='%1.1f%%', colors=colors, startangle=90)
ax.set_title('Watch Time by Device Category', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Content Analysis

In [ ]:
# Series vs Movies (rough detection)
def is_series(title):
    title_lower = str(title).lower()
    return 'season' in title_lower or 'episode' in title_lower or ': season' in title_lower

main_content['IsSeries'] = main_content['Title'].apply(is_series)

series_hours = main_content[main_content['IsSeries']]['Duration_Hours'].sum()
movie_hours = main_content[~main_content['IsSeries']]['Duration_Hours'].sum()
total = series_hours + movie_hours

print('Content Type Distribution:')
print('=' * 40)
print(f'Series: {series_hours:.0f} hours ({series_hours/total*100:.0f}%)')
print(f'Movies/Standalone: {movie_hours:.0f} hours ({movie_hours/total*100:.0f}%)')

In [ ]:
# Top 10 most watched titles
top_titles = main_content.groupby('Title')['Duration_Hours'].sum().sort_values(ascending=False).head(10)

print('\nTop 10 Most Watched Titles:')
print('=' * 50)
for i, (title, hours) in enumerate(top_titles.items(), 1):
    print(f'{i}. {title[:45]}: {hours:.2f} hours')

In [ ]:
# Visualization: Top 10 Titles
fig, ax = plt.subplots(figsize=(12, 8))
y_pos = range(len(top_titles))
ax.barh(y_pos, top_titles.values, color=sns.color_palette('husl', len(top_titles)))
ax.set_yticks(y_pos)
ax.set_yticklabels([t[:40] + '...' if len(t) > 40 else t for t in top_titles.index])
ax.set_xlabel('Watch Time (hours)', fontsize=12)
ax.set_title('Top 10 Most Watched Titles', fontsize=14)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Geographic Analysis

In [ ]:
# Viewing by country
country_watch = main_content.groupby('Country')['Duration_Hours'].sum().sort_values(ascending=False)

print('Viewing by Country:')
print('=' * 40)
for country, hours in country_watch.items():
    pct = hours / country_watch.sum() * 100
    print(f'  {country}: {hours:.0f} hours ({pct:.1f}%)')

In [ ]:
# Visualization: Country Distribution
fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('husl', len(country_watch))
bars = ax.bar(range(len(country_watch)), country_watch.values, color=colors)
ax.set_xticks(range(len(country_watch)))
ax.set_xticklabels([c[:20] for c in country_watch.index], rotation=45, ha='right')
ax.set_ylabel('Watch Time (hours)', fontsize=12)
ax.set_title('Netflix Viewing by Country', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Session Patterns

In [ ]:
# Sessions per day analysis
daily_sessions = main_content.groupby(main_content['Start Time'].dt.date).size()

print('Session Statistics:')
print('=' * 40)
print(f'Average sessions per day: {daily_sessions.mean():.1f}')
print(f'Max sessions in a day: {daily_sessions.max()}')
print(f'Min sessions in a day: {daily_sessions.min()}')
print(f'Days with viewing: {len(daily_sessions)}')

# Binge sessions (>2 hours)
binge_sessions = main_content[main_content['Duration_Seconds'] > 7200]
print(f'\nBinge sessions (>2 hours): {len(binge_sessions)} ({len(binge_sessions)/len(main_content)*100:.1f}%)')

In [ ]:
# Session duration distribution
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(main_content['Duration_Minutes'], bins=50, edgecolor='black', alpha=0.7, color='#4ECDC4')
ax.axvline(main_content['Duration_Minutes'].mean(), color='red', linestyle='--', 
           label=f'Mean: {main_content["Duration_Minutes"].mean():.1f} min')
ax.set_xlabel('Session Duration (minutes)', fontsize=12)
ax.set_ylabel('Number of Sessions', fontsize=12)
ax.set_title('Distribution of Session Durations', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

## 10. Summary & Key Insights

### Viewing Behavior Insights:

1. **Total Consumption**: 1,687 hours of content across 4,454 sessions
2. **Peak Viewing Time**: 5 PM is the peak hour with 8.5% of all viewing
3. **Device Preference**: Mac/Safari dominates at 75.3%, followed by iPhone at 23.2%
4. **Content Mix**: 70% series vs 30% movies by watch time
5. **Geographic Split**: US (1,281 hrs) vs India (389 hrs) as primary viewing locations
6. **Session Behavior**: Average session is 25.7 minutes with 6.1 sessions per day
7. **Weekend Effect**: 1.2x higher viewing on weekends vs weekdays